# Prediction Confidence

Analyze prediction confidence mechanics: calibration, overconfidence detection,
confidence sources by component, layerwise confidence evolution, and entropy decomposition.

References:
- Kadavath et al. (2022) "Language Models (Mostly) Know What They Know"
- Guo et al. (2017) "On Calibration of Modern Neural Networks"

## Why This Matters

Prediction confidence analyzes how the model arrives at its output predictions. Tracking prediction formation across layers reveals the computational process — when the model commits to an answer, what changes its mind, and how confident it is.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_confidence import (
    confidence_profile,
    layerwise_confidence_evolution,
    confidence_source_attribution,
    entropy_decomposition,
    confidence_calibration,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Confidence Profile

Profile prediction confidence at each token position: top-1 probability,
predicted token, entropy, and overconfident positions.

In [ ]:
result = confidence_profile(model, tokens)
print(f"Mean confidence: {result['confidence_mean']:.4f}")
print(f"\nPer-position confidence:")
for i in range(len(tokens)):
    print(f"  Pos {i}: top1_prob={result['top1_probs'][i]:.4f}, "
          f"token={result['top1_tokens'][i]}, entropy={result['entropy'][i]:.4f}")
print(f"\nOverconfident positions (>0.9): {result['overconfident_positions']}")

## Layerwise Confidence Evolution

Track how prediction confidence builds up through the layers using logit lens.
Identifies the "confidence emergence layer" where the model first becomes confident.

In [ ]:
result = layerwise_confidence_evolution(model, tokens)
print("Layer-by-layer confidence evolution:")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: entropy={result['layer_entropy'][i]:.4f}, "
          f"top1_prob={result['layer_top1_prob'][i]:.4f}")
print(f"\nConfidence emergence layer: {result['confidence_emergence_layer']}")
print(f"Entropy reduction per layer: {result['entropy_reduction']}")

## Confidence Source Attribution

Identify which components (attention heads, MLPs) boost or reduce prediction
confidence by measuring entropy change when each is ablated.

In [ ]:
result = confidence_source_attribution(model, tokens)
print("Component confidence effects (entropy change when ablated):")
for comp, effect in sorted(result["component_confidence_effects"].items(), key=lambda x: -abs(x[1])):
    sign = "+" if effect > 0 else ""
    print(f"  {comp}: {sign}{effect:.4f}")
print(f"\nConfidence boosters: {result['confidence_boosters']}")
print(f"Confidence reducers: {result['confidence_reducers']}")
print(f"\nTop token logit attributions:")
for comp, attr in sorted(result["top_token_attributions"].items(), key=lambda x: -abs(x[1])):
    print(f"  {comp}: {attr:.4f}")

## Entropy Decomposition

Decompose the final prediction entropy into contributions from embedding,
attention, and MLP components.

In [ ]:
result = entropy_decomposition(model, tokens)
print(f"Total entropy: {result['total_entropy']:.4f}")
print(f"Entropy from embedding: {result['entropy_from_embedding']:.4f}")
print(f"Entropy from attention: {result['entropy_from_attention']:.4f}")
print(f"Entropy from MLP: {result['entropy_from_mlp']:.4f}")
print(f"\nPer-component contributions:")
for comp, val in result["component_entropy_contributions"].items():
    print(f"  {comp}: {val:.4f}")

## Confidence Calibration

Measure how well the model's confidence predicts correctness across multiple
inputs. A well-calibrated model has confidence close to accuracy.

In [ ]:
tokens_list = [
    jnp.array([0, 5, 10, 15, 20]),
    jnp.array([1, 2, 3, 4, 5]),
    jnp.array([10, 20, 30, 40, 50]),
    jnp.array([3, 6, 9, 12, 15]),
]
correct_tokens = [0, 0, 0, 0]  # arbitrary for demo
result = confidence_calibration(model, tokens_list, correct_tokens)
print(f"Mean confidence: {result['mean_confidence']:.4f}")
print(f"Accuracy: {result['accuracy']:.4f}")
print(f"Calibration error: {result['calibration_error']:.4f}")
print(f"\nPer-input:")
for i, (conf, corr) in enumerate(zip(result["confidences"], result["correct"])):
    print(f"  Input {i}: confidence={conf:.4f}, correct={corr}")